# Stack Overflow Q&A Search


## 1. Install Dependencies

In [1]:
%pip install pandas numpy sentence-transformers deep-translator faiss-cpu scikit-learn beautifulsoup4 gdown transformers -q


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import Libraries

In [2]:
import pandas as pd
import numpy as np
import faiss
import pickle
import re
import json
import logging
import gdown
import os
from bs4 import BeautifulSoup
from functools import lru_cache

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.utils.validation import check_is_fitted

from sentence_transformers import SentenceTransformer
from deep_translator import GoogleTranslator

import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('so_search')


## 3. Load & Prepare Dataset

In [3]:
import os

# Jalur pencarian dataset lokal secara relatif
possible_paths = [
    os.path.join('..', 'Dataset', 'dataset.csv'),
    os.path.join('Dataset', 'dataset.csv'),
    'dataset.csv'
]

output = None
for path in possible_paths:
    if os.path.exists(path):
        output = path
        break

if output is None:
    raise FileNotFoundError("Berkas dataset.csv tidak ditemukan di direktori lokal.")

logger.info(f'Loading local dataset from: {output}')
df = pd.read_csv(output)
logger.info(f'Dataset shape: {df.shape}')
df.head(5)

14:43:06 [INFO] Loading local dataset from: ..\Dataset\dataset.csv
14:43:08 [INFO] Dataset shape: (33628, 8)


,Id,Title,Body,Score,Tags,AnswerId,AnswerBody,AnswerScore
0,260,Adding scripting functionality to .NET applica...,<p>I have a little game written in C#. It uses...,49,c#,307.0,"<p><a href=""http://www.codeproject.com/Article...",28.0
1,330,Should I use nested classes in this case?,<p>I am working on a collection of classes use...,29,c++,332.0,<p>I would be a bit reluctant to use nested cl...,19.0
2,650,Automatically update version number,<p>I would like the version property of my app...,79,c#,655.0,"<p>With the ""Built in"" stuff, you can't, as us...",69.0
3,930,How do I connect to a database and loop over a...,<p>What's the simplest way to connect and quer...,28,c#,951.0,<p>@Goyuix -- that's excellent for something w...,26.0
4,1010,"How to get the value of built, encoded ViewState?",<p>I need to grab the base64-encoded represent...,14,c#,1074.0,"<p>Rex, I suspect a good place to start lookin...",7.0


In [4]:
def clean_html(text):
    if pd.isna(text):
        return ''
    text = BeautifulSoup(str(text), 'html.parser').get_text(separator=' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Title']       = df['Title'].fillna('').apply(clean_html)
df['Body']        = df['Body'].fillna('').apply(clean_html)
df['AnswerBody']  = df['AnswerBody'].fillna('').apply(clean_html)
df['AnswerScore'] = pd.to_numeric(df['AnswerScore'], errors='coerce').fillna(0)

VALID_TAGS = [
    'assembly', 'c#', 'c++', 'dart', 'go', 'haskell', 'java',
    'javascript', 'kotlin', 'lua', 'objective-c', 'perl', 'php',
    'python', 'r', 'ruby', 'rust', 'scala', 'swift', 'typescript'
]

def extract_primary_tag(tag_str):
    if pd.isna(tag_str):
        return 'other'
    parts = str(tag_str).strip().lower().split()
    for part in parts:
        if part in VALID_TAGS:
            return part
    return 'other'

df['Tags'] = df['Tags'].apply(extract_primary_tag)

before = len(df)
df = df[df['Tags'] != 'other'].reset_index(drop=True)
logger.info(f'Baris dihapus (tag tidak valid): {before - len(df)}')

df['combined_text'] = (
    df['Title'].astype(str) + ' ' +
    df['Title'].astype(str) + ' ' +
    df['Title'].astype(str) + ' ' +
    df['Body'].astype(str)
)

df = df.reset_index(drop=True)
logger.info(f'Total rows: {len(df)}')
logger.info(f'Total tags unik: {df["Tags"].nunique()}')


14:44:08 [INFO] Baris dihapus (tag tidak valid): 0
14:44:08 [INFO] Total rows: 33628
14:44:08 [INFO] Total tags unik: 20


## 4. Daftar Tags yang Tersedia

In [5]:
all_tags = sorted(df['Tags'].unique().tolist())
logger.info(f'Total tags: {len(all_tags)}')
print(f'\nTotal tags: {len(all_tags)}')
print('Tags dan jumlah data:')
for t in all_tags:
    count = len(df[df['Tags'] == t])
    print(f'  {t:20s}  {count:>6} rows')


14:44:08 [INFO] Total tags: 20



Total tags: 20
Tags dan jumlah data:
  assembly                2000 rows
  c#                      2087 rows
  c++                     2118 rows
  dart                     613 rows
  go                      1838 rows
  haskell                 1985 rows
  java                    2231 rows
  javascript              2554 rows
  kotlin                   132 rows
  lua                      911 rows
  objective-c             2058 rows
  perl                    1947 rows
  php                     1902 rows
  python                  1996 rows
  r                       1942 rows
  ruby                    1903 rows
  rust                     516 rows
  scala                   1795 rows
  swift                   1831 rows
  typescript              1269 rows


## 5. Build TF-IDF Model

In [6]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=30000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(df['combined_text'])
logger.info(f'TF-IDF shape: {tfidf_matrix.shape}')
logger.info(f'Vocabulary size: {len(tfidf.vocabulary_)}')


14:44:32 [INFO] TF-IDF shape: (33628, 30000)
14:44:32 [INFO] Vocabulary size: 30000


## 6. Build SBERT Semantic Embedding

In [7]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
SBERT_MAX_TOKENS = 256

embeddings = embedding_model.encode(
    df['combined_text'].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    batch_size=64
)
logger.info(f'Embedding shape: {embeddings.shape}')

14:44:32 [INFO] No device provided, using cpu
14:44:33 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
14:44:33 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
14:44:33 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
14:44:33 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
14:44:33 [INFO] Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
14:44:34 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/confi

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

14:44:36 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
14:44:36 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
14:44:36 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
14:44:37 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
14:44:37 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
14:44:37 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/tokenizer_config.json "HTTP/

Batches:   0%|          | 0/526 [00:00<?, ?it/s]

15:28:09 [INFO] Embedding shape: (33628, 384)


## 7. Build FAISS Index (Global + Per-Tag untuk Tag Kecil)

In [8]:
dimension = embeddings.shape[1]
embeddings_norm = embeddings.copy().astype('float32')
faiss.normalize_L2(embeddings_norm)

index = faiss.IndexFlatIP(dimension)
index.add(embeddings_norm)
logger.info(f'Global FAISS indexed: {index.ntotal} vectors')

SMALL_TAG_THRESHOLD = 500
tag_local_indices = {}

small_tags = [t for t in df['Tags'].unique() if len(df[df['Tags'] == t]) < SMALL_TAG_THRESHOLD]

for tag in small_tags:
    tag_mask = df['Tags'] == tag
    tag_global_idx = df[tag_mask].index.tolist()
    sub_emb = embeddings_norm[tag_global_idx].copy()
    local_idx = faiss.IndexFlatIP(dimension)
    local_idx.add(sub_emb)
    tag_local_indices[tag] = (local_idx, tag_global_idx)


15:28:09 [INFO] Global FAISS indexed: 33628 vectors


## 8. Fungsi Helper

In [9]:
translator = GoogleTranslator(source='auto', target='en')

def translate_if_needed(text: str) -> str:
    try:
        result = translator.translate(text)
        return result if result else text
    except Exception as e:
        logger.warning(f'Translasi gagal: {e}. Menggunakan query asli.')
        return text

def normalize_score(scores_array: np.ndarray) -> np.ndarray:
    mn, mx = scores_array.min(), scores_array.max()
    if mx == mn:
        return np.ones_like(scores_array)
    return (scores_array - mn) / (mx - mn)

def check_query_length(text: str, max_tokens: int = SBERT_MAX_TOKENS) -> str:
    words = text.split()
    estimated_tokens = int(len(words) * 1.4)
    if estimated_tokens > max_tokens:
        logger.warning(
            f'Query terlalu panjang (~{estimated_tokens} token estimasi, '
            f'batas SBERT={max_tokens}). '
            f'Query dipotong pada {max_tokens} token pertama.'
        )
        safe_words = max_tokens // 2
        text = ' '.join(words[:safe_words])
    return text

@lru_cache(maxsize=256)
def _encode_query_cached(query_text: str) -> np.ndarray:
    emb = embedding_model.encode([query_text], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(emb)
    return emb.copy()

@lru_cache(maxsize=256)
def _tfidf_transform_cached(query_text: str):
    return tfidf.transform([query_text])


## 9. Fungsi Pencarian Hybrid

In [10]:
def search_hybrid(
    query: str,
    tag_filter: str = None,
    top_k: int = 5,
    alpha: float = 0.4,
    beta: float = 0.5,
    gamma: float = 0.1
) -> list[dict]:

    if tag_filter:
        tag_filter_clean = tag_filter.strip().lower()

        if tag_filter_clean not in VALID_TAGS:
            logger.warning(
                f'Tag "{tag_filter_clean}" tidak ada di VALID_TAGS. '
                f'Tag tersedia: {VALID_TAGS}'
            )
            return []

        mask = df['Tags'] == tag_filter_clean
        subset_df = df[mask].copy()
        subset_indices = df[mask].index.tolist()

        if len(subset_df) == 0:
            logger.warning(f'Tag "{tag_filter_clean}" ada di VALID_TAGS tapi tidak ada data.')
            return []
    else:
        tag_filter_clean = None
        subset_df = df
        subset_indices = list(range(len(df)))

    translated_query = translate_if_needed(query)
    translated_query = check_query_length(translated_query)

    logger.info(f'Query  (asli)      : {query[:80]}')
    logger.info(f'Query  (translated): {translated_query[:80]}')
    logger.info(f'Scope              : {len(subset_df)} rows (tag="{tag_filter_clean}")')

    query_vec       = _tfidf_transform_cached(translated_query)
    tfidf_scores_all = cosine_similarity(query_vec, tfidf_matrix).flatten()
    tfidf_scores    = tfidf_scores_all[subset_indices]

    if tag_filter_clean and tag_filter_clean in tag_local_indices:
        local_faiss_idx, local_global_indices = tag_local_indices[tag_filter_clean]
        query_emb = _encode_query_cached(translated_query)
        k_local = min(local_faiss_idx.ntotal, top_k * 5)
        scores_local, indices_local = local_faiss_idx.search(query_emb, k_local)

        sbert_scores = np.zeros(len(subset_indices))
        for sc, loc_idx in zip(scores_local[0], indices_local[0]):
            if loc_idx >= 0:
                global_idx = local_global_indices[loc_idx]
                if global_idx in subset_indices:
                    pos = subset_indices.index(global_idx)
                    sbert_scores[pos] = float(sc)

    else:
        query_emb = _encode_query_cached(translated_query)
        k_search = min(len(df), top_k * 20)
        sbert_scores_all = np.zeros(len(df))
        scores_faiss, indices_faiss = index.search(query_emb, k_search)
        for sc, idx in zip(scores_faiss[0], indices_faiss[0]):
            if idx >= 0:
                sbert_scores_all[idx] = float(sc)
        sbert_scores = sbert_scores_all[subset_indices]

    answer_scores = subset_df['AnswerScore'].values.astype(float)
    answer_scores = np.log1p(np.clip(answer_scores, 0, None))

    tfidf_norm  = normalize_score(tfidf_scores)
    sbert_norm  = normalize_score(sbert_scores)
    answer_norm = normalize_score(answer_scores)

    fusion = alpha * tfidf_norm + beta * sbert_norm + gamma * answer_norm
    top_local_indices = fusion.argsort()[-top_k:][::-1]

    results = []
    for local_idx in top_local_indices:
        global_idx = subset_indices[local_idx]
        row = df.iloc[global_idx]
        results.append({
            'fusion_score':    float(fusion[local_idx]),
            'tfidf_score':     float(tfidf_norm[local_idx]),
            'sbert_score':     float(sbert_norm[local_idx]),
            'answer_score_raw': int(row['AnswerScore']),
            'title':  row['Title'],
            'tags':   row['Tags'],
            'answer': row['AnswerBody'],
        })

    return results


## 10. Testing dengan Filter Tag

In [11]:
query = 'BAGAIMANA CARA PHYTON TERHUBUNG DENGAN MYSQL'
tag   = 'python'

results = search_hybrid(query, tag_filter=tag, top_k=5)

print('='*70)
print(f'HYBRID RESULTS  |  query="{query}"  |  tag="{tag}"')
print('='*70)
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['fusion_score']:.4f}] {r['title']}")
    print(f"   tfidf={r['tfidf_score']:.3f} | sbert={r['sbert_score']:.3f} | ans_score={r['answer_score_raw']}")
    print(f"   tags: {r['tags']}")
    print(f"   answer: {r['answer'][:300]}...")
    print('-'*70)


15:28:14 [INFO] Query  (asli)      : BAGAIMANA CARA PHYTON TERHUBUNG DENGAN MYSQL
15:28:14 [INFO] Query  (translated): HOW DOES PHYTON CONNECT WITH MYSQL
15:28:14 [INFO] Scope              : 1996 rows (tag="python")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

HYBRID RESULTS  |  query="BAGAIMANA CARA PHYTON TERHUBUNG DENGAN MYSQL"  |  tag="python"
1. [0.9372] Python 3.3 Mysql Connector
   tfidf=1.000 | sbert=1.000 | ans_score=14
   tags: python
   answer: There is a module called Pymysql which you may like: """This pure Python MySQL client provides a DB-API to a MySQL database by talking directly to the server via the binary client/server protocol.""" import pymysql conn = pymysql.connect(host='127.0.0.1', unix_socket='/tmp/mysql.sock', user='root', ...
----------------------------------------------------------------------
2. [0.6375] Does Python support MySQL prepared statements?
   tfidf=0.310 | sbert=0.924 | ans_score=42
   tags: python
   answer: Most languages provide a way to do generic parameterized statements, Python is no different. When a parameterized query is used databases that support preparing statements will automatically do so. In python a parameterized query looks like this: cursor.execute("SELECT FROM tablename WHERE field

In [12]:
query2 = 'difference between machine code and assembly'
tag2   = 'assembly'

results2 = search_hybrid(query2, tag_filter=tag2, top_k=3)

print('='*70)
print(f'HYBRID RESULTS  |  query="{query2}"  |  tag="{tag2}"')
print('='*70)
for i, r in enumerate(results2, 1):
    print(f"{i}. [{r['fusion_score']:.4f}] {r['title']}")
    print(f"   answer: {r['answer'][:300]}...")
    print('-'*70)


15:28:17 [INFO] Query  (asli)      : difference between machine code and assembly
15:28:17 [INFO] Query  (translated): difference between machine code and assembly
15:28:17 [INFO] Scope              : 2000 rows (tag="assembly")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

HYBRID RESULTS  |  query="difference between machine code and assembly"  |  tag="assembly"
1. [1.0000] Assembly code vs Machine code vs Object code?
   answer: Machine code is binary (1's and 0's) code that can be executed directly by the cpu. If you were to open a "machine code" file in a text editor you would see garbage, including unprintable characters (no, not those unprintable characters ;). Object code is a portion of machine code that hasn't yet be...
----------------------------------------------------------------------
2. [0.6000] What is the actual relation between assembly, machine code, bytecode, and opcode?
   answer: Is there some sort of standard reference that lists out all of those numbers, and what they mean, for whatever architecture you are on, and how each set of numbers maps to each assembly instruction? Yes, though they can be very complex. Also, due to the prevalence of assemblers and compilers, they'r...
--------------------------------------------------------

In [13]:
query3 = 'how to sort a list'
results3 = search_hybrid(query3, tag_filter=None, top_k=5)

print('='*70)
print(f'GLOBAL RESULTS (tanpa filter tag) | query="{query3}"')
print('='*70)
for i, r in enumerate(results3, 1):
    print(f"{i}. [{r['fusion_score']:.4f}] [{r['tags']:12s}] {r['title']}")
    print('-'*70)


15:28:19 [INFO] Query  (asli)      : how to sort a list
15:28:19 [INFO] Query  (translated): how to sort a list
15:28:19 [INFO] Scope              : 33628 rows (tag="None")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

GLOBAL RESULTS (tanpa filter tag) | query="how to sort a list"
1. [0.9107] [python      ] How does Python sort a list of tuples?
----------------------------------------------------------------------
2. [0.8347] [java        ] How to sort a HashSet?
----------------------------------------------------------------------
3. [0.8241] [python      ] How to correctly sort a string with a number inside?
----------------------------------------------------------------------
4. [0.7652] [c#          ] How to Sort a List<> by a Integer stored in the struct my List<> holds
----------------------------------------------------------------------
5. [0.7616] [java        ] Sort List of Strings with Localization
----------------------------------------------------------------------


In [14]:
print('Test tag tidak valid (cobol):')
results4 = search_hybrid('print hello world', tag_filter='cobol', top_k=3)
print(f'Jumlah hasil: {len(results4)}')

print('\nTest query pendek:')
results5 = search_hybrid('error', tag_filter='python', top_k=3)
print(f'Jumlah hasil: {len(results5)}')


15:28:19 [WARNING] Tag "cobol" tidak ada di VALID_TAGS. Tag tersedia: ['assembly', 'c#', 'c++', 'dart', 'go', 'haskell', 'java', 'javascript', 'kotlin', 'lua', 'objective-c', 'perl', 'php', 'python', 'r', 'ruby', 'rust', 'scala', 'swift', 'typescript']


Test tag tidak valid (cobol):
Jumlah hasil: 0

Test query pendek:


15:28:20 [INFO] Query  (asli)      : error
15:28:20 [INFO] Query  (translated): error
15:28:20 [INFO] Scope              : 1996 rows (tag="python")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Jumlah hasil: 3


In [15]:
long_query = (
    'I am trying to build a web application using Python and Django framework '
    'but I keep getting a database connection error every time I try to query '
    'the MySQL database. The error message says connection refused on port 3306 '
    'and I have already checked the firewall settings and the MySQL service '
    'is running. I have also verified the username and password are correct '
    'and the database exists. The connection string seems right but it still '
    'fails when I run the application in production environment on Ubuntu server.'
)
print(f'Query length: {len(long_query.split())} words')
results6 = search_hybrid(long_query, tag_filter='python', top_k=3)
print(f'Jumlah hasil: {len(results6)}')


Query length: 87 words


15:28:21 [INFO] Query  (asli)      : I am trying to build a web application using Python and Django framework but I k
15:28:21 [INFO] Query  (translated): I am trying to build a web application using Python and Django framework but I k
15:28:21 [INFO] Scope              : 1996 rows (tag="python")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Jumlah hasil: 3


In [16]:
for small_tag in ['haskell', 'lua', 'scala']:
    if small_tag in df['Tags'].values:
        r = search_hybrid('how to define a function', tag_filter=small_tag, top_k=3)
        sbert_scores = [x['sbert_score'] for x in r]
        print(f'[{small_tag}] {len(r)} hasil | avg sbert_score={sum(sbert_scores)/len(sbert_scores):.3f}')
    else:
        print(f'[{small_tag}] tidak ada di dataset, skip.')


15:28:22 [INFO] Query  (asli)      : how to define a function
15:28:22 [INFO] Query  (translated): how to define a function
15:28:22 [INFO] Scope              : 1985 rows (tag="haskell")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[haskell] 3 hasil | avg sbert_score=0.963


15:28:23 [INFO] Query  (asli)      : how to define a function
15:28:23 [INFO] Query  (translated): how to define a function
15:28:23 [INFO] Scope              : 911 rows (tag="lua")


[lua] 3 hasil | avg sbert_score=0.659


15:28:25 [INFO] Query  (asli)      : how to define a function
15:28:25 [INFO] Query  (translated): how to define a function
15:28:25 [INFO] Scope              : 1795 rows (tag="scala")


[scala] 3 hasil | avg sbert_score=0.646


## 11. Unit Test & Evaluasi Kuantitatif

In [17]:
EVAL_QUERIES = [
    {
        'query': 'how to connect python to mysql database',
        'tag': 'python',
        'expected_keywords': ['mysql', 'connect', 'database', 'pymysql', 'sqlalchemy', 'cursor']
    },
    {
        'query': 'difference between machine code and assembly language',
        'tag': 'assembly',
        'expected_keywords': ['assembly', 'machine', 'instruction', 'code', 'processor']
    },
    {
        'query': 'how to sort array in javascript',
        'tag': 'javascript',
        'expected_keywords': ['sort', 'array', 'javascript', 'function', 'method']
    },
    {
        'query': 'null pointer exception java',
        'tag': 'java',
        'expected_keywords': ['null', 'pointer', 'exception', 'object', 'reference']
    },
    {
        'query': 'how to read file in python',
        'tag': 'python',
        'expected_keywords': ['file', 'open', 'read', 'with', 'lines']
    },
]

def compute_keyword_hit(answer_text: str, keywords: list) -> bool:
    answer_lower = answer_text.lower()
    return any(kw.lower() in answer_lower for kw in keywords)

def evaluate_model(eval_queries: list, top_k: int = 5) -> dict:
    results_summary = []
    hits = 0

    print(f'\n{"="*70}')
    print(f'EVALUASI MODEL — {len(eval_queries)} query, top_k={top_k}')
    print(f'{"="*70}')

    for i, eq in enumerate(eval_queries, 1):
        results = search_hybrid(eq['query'], tag_filter=eq['tag'], top_k=top_k)

        if not results:
            hit = False
            best_score = 0.0
        else:
            combined_answers = ' '.join([r['answer'] + ' ' + r['title'] for r in results])
            hit = compute_keyword_hit(combined_answers, eq['expected_keywords'])
            best_score = results[0]['fusion_score'] if results else 0.0

        if hit:
            hits += 1

        status = 'HIT' if hit else 'MISS'
        print(f'\n[Q{i}] {status}')
        print(f'  Query   : {eq["query"]}')
        print(f'  Tag     : {eq["tag"]}')
        print(f'  Keywords: {eq["expected_keywords"]}')
        print(f'  Top hasil: {results[0]["title"] if results else "—"}')
        print(f'  Best fusion_score: {best_score:.4f}')

        results_summary.append({
            'query': eq['query'],
            'tag': eq['tag'],
            'hit': hit,
            'best_fusion_score': best_score,
            'num_results': len(results)
        })

    keyword_hit_rate = hits / len(eval_queries)
    avg_score = np.mean([r['best_fusion_score'] for r in results_summary])

    print(f'\n{"="*70}')
    print(f'RINGKASAN EVALUASI')
    print(f'{"="*70}')
    print(f'  Keyword Hit Rate @{top_k} : {keyword_hit_rate:.2%} ({hits}/{len(eval_queries)})')
    print(f'  Avg Fusion Score       : {avg_score:.4f}')
    print(f'{"="*70}')

    return {
        'keyword_hit_rate': keyword_hit_rate,
        'avg_fusion_score': avg_score,
        'top_k': top_k,
        'n_queries': len(eval_queries),
        'detail': results_summary
    }

eval_metrics = evaluate_model(EVAL_QUERIES, top_k=5)



EVALUASI MODEL — 5 query, top_k=5


15:28:26 [INFO] Query  (asli)      : how to connect python to mysql database
15:28:26 [INFO] Query  (translated): how to connect python to mysql database
15:28:26 [INFO] Scope              : 1996 rows (tag="python")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


[Q1] HIT
  Query   : how to connect python to mysql database
  Tag     : python
  Keywords: ['mysql', 'connect', 'database', 'pymysql', 'sqlalchemy', 'cursor']
  Top hasil: Python 3.3 Mysql Connector
  Best fusion_score: 0.9372


15:28:27 [INFO] Query  (asli)      : difference between machine code and assembly language
15:28:27 [INFO] Query  (translated): difference between machine code and assembly language
15:28:27 [INFO] Scope              : 2000 rows (tag="assembly")


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


[Q4] HIT
  Query   : null pointer exception java
  Tag     : java
  Keywords: ['null', 'pointer', 'exception', 'object', 'reference']
  Top hasil: Is Catching a Null Pointer Exception a Code Smell?
  Best fusion_score: 0.9383


15:28:33 [INFO] Query  (asli)      : how to read file in python
15:28:33 [INFO] Query  (translated): how to read file in python
15:28:33 [INFO] Scope              : 1996 rows (tag="python")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


[Q5] HIT
  Query   : how to read file in python
  Tag     : python
  Keywords: ['file', 'open', 'read', 'with', 'lines']
  Top hasil: Reading entire file in Python
  Best fusion_score: 0.9780

RINGKASAN EVALUASI
  Keyword Hit Rate @5 : 100.00% (5/5)
  Avg Fusion Score       : 0.9345
